# 05 — Deeper Root-Cause Investigation
### Does lateness fully explain poor reviews? And what else is actually going on?

Notebooks 3–4 treated "poor reviews" as essentially downstream of "late delivery." That link is real (on-time orders average a 4.29 review score vs. 2.57 for late ones) — but it was only checked at a state-aggregated level, and a large number of **on-time orders still get 1–2 star reviews**. That gap means part of the original business question — *what's driving poor reviews* — was left half-answered.

This notebook does two things:
1. **Tests the lateness → reviews link directly at the order level**, then isolates and investigates the on-time-but-bad-review segment specifically
2. **Goes looking for additional real issues** in the data beyond the original scope, tests each one with actual numbers, and reports what holds up and what doesn't — including a hypothesis that turned out to be a **null result**, because knowing what *isn't* a driver is as useful as knowing what is

Structure:
- Section 1: Order-level test of lateness vs. review score
- Section 2: What actually differs about on-time orders with bad reviews (price, freight, item count)
- Section 3: Mining the review comment text itself for concrete complaint signals
- Section 4: A revenue-loss channel the earlier notebooks never covered — orders that never deliver at all
- Section 5: A hypothesis tested and ruled out — payment approval delay
- Section 6: Updated findings summary


In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DB_PATH = '../Data/processed/olist.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)


Connected to ../Data/processed/olist.db


## Section 1 — Does lateness actually explain poor reviews? (order-level test)

Notebook 3 only checked this at the **state level** (average delivery days vs. average review score, across ~20 states) — that's an aggregate correlation, and aggregate correlations can look stronger than the underlying individual relationship really is. The honest test is at the **order level**.

In [2]:
order_level = pd.read_sql("""
    SELECT delay_days, delivery_time_days, review_score, is_late
    FROM master_orders_sql
    WHERE is_delivered = 1 AND review_score IS NOT NULL AND delay_days IS NOT NULL
""", conn)

corr_delay = order_level['delay_days'].corr(order_level['review_score'])
corr_time = order_level['delivery_time_days'].corr(order_level['review_score'])

avg_by_late = order_level.groupby('is_late')['review_score'].agg(['mean', 'count']).round(3)

print(f'Order-level correlation, delay_days vs review_score:        {corr_delay:.3f}')
print(f'Order-level correlation, delivery_time_days vs review_score: {corr_time:.3f}')
print()
print('Average review score, on-time (0) vs late (1):')
avg_by_late


Order-level correlation, delay_days vs review_score:        -0.267
Order-level correlation, delivery_time_days vs review_score: -0.334

Average review score, on-time (0) vs late (1):


,mean,count
is_late,,
0,4.294,88163
1,2.565,7661


**Verdict: the link is real, but partial.** On-time orders average **4.29** stars vs. **2.57** for late ones — a large, meaningful gap that holds at the order level, not just in aggregate. But the correlation coefficients (−0.27 for delay, −0.33 for total delivery time) are moderate, not dominant. Lateness is a real driver of poor reviews — it is not the *only* one. The remaining question is what explains the reviews that lateness doesn't.

In [3]:
review_dist = pd.read_sql("""
    SELECT is_late, review_score, COUNT(*) AS n
    FROM master_orders_sql
    WHERE is_delivered = 1 AND review_score IS NOT NULL
    GROUP BY is_late, review_score
    ORDER BY is_late, review_score
""", conn)
pivot = review_dist.pivot(index='review_score', columns='is_late', values='n')
pivot.columns = ['on_time', 'late']
print('Review score distribution, on-time vs late:')
print(pivot)
print()
n_on_time_bad = int(pivot.loc[[1,2], 'on_time'].sum())
print(f'On-time orders that STILL got a 1-2 star review: {n_on_time_bad:,}')


Review score distribution, on-time vs late:
              on_time  late
review_score               
1                5812  3540
2                2319   602
3                7044   872
4               17944   944
5               55052  1703

On-time orders that STILL got a 1-2 star review: 8,131


That last number is the actual target for the rest of this section: thousands of orders where delivery worked fine, and the customer was still unhappy. Lateness doesn't explain these — something else does.

## Section 2 — What actually differs about on-time orders with bad reviews?

Two competing hypotheses, both testable with columns already in `master_orders_sql`:
- **H1 — shipping cost frustration:** customers are unhappy about what they paid for freight relative to the item, independent of speed
- **H2 — higher-stakes orders have more that can go wrong:** pricier, multi-item orders are more likely to have a wrong/damaged/missing item, a mismatch with expectations, etc., regardless of how fast they arrived

In [4]:
segment_compare = pd.read_sql("""
    SELECT
        CASE WHEN review_score <= 2 THEN 'bad' WHEN review_score >= 4 THEN 'good' ELSE 'mid' END AS review_group,
        COUNT(*) AS n_orders,
        ROUND(AVG(item_price), 2) AS avg_item_price,
        ROUND(AVG(freight_value), 2) AS avg_freight,
        ROUND(AVG(freight_value * 1.0 / NULLIF(item_price, 0)), 3) AS avg_freight_ratio,
        ROUND(AVG(n_items), 2) AS avg_n_items
    FROM master_orders_sql
    WHERE is_delivered = 1 AND is_late = 0 AND review_score IS NOT NULL
    GROUP BY review_group
""", conn)
segment_compare


,review_group,n_orders,avg_item_price,avg_freight,avg_freight_ratio,avg_n_items
0,bad,8131,162.64,28.80,0.320,1.41
1,good,72996,133.88,21.84,0.304,1.11
2,mid,7044,126.40,23.46,0.328,1.17


**H1 (shipping cost) doesn't hold up:** the freight-to-price ratio is essentially identical across bad (0.320), mid (0.328), and good (0.304) reviews — a ~0.02 difference is noise, not a driver. Ruling this out matters as much as confirming a real one.

**H2 (higher-stakes orders) does hold up:** on-time orders with bad reviews average **R$162.64** per item vs. **R$133.88** for good reviews (+21%), and **1.41 items per order** vs. **1.11** (+27%). Bigger, pricier, multi-item orders are meaningfully more likely to disappoint even when delivery is fast — consistent with more surface area for something to go wrong (wrong item, partial fulfillment, product/description mismatch, damage in a bigger package) rather than a delivery-speed complaint.

## Section 3 — Mining the review text itself for a direct signal

The structured columns can show *which* orders are more likely to get a bad review, but not *why* — that's in the free-text `review_comment_message`. Before mining word content, first check whether bad-review customers even bother writing a comment.

In [5]:
comment_rates = pd.read_sql("""
    SELECT
        CASE WHEN r.review_score <= 2 THEN 'bad' WHEN r.review_score >= 4 THEN 'good' ELSE 'mid' END AS review_group,
        COUNT(*) AS n,
        SUM(CASE WHEN r.review_comment_message IS NOT NULL AND LENGTH(TRIM(r.review_comment_message)) > 0 THEN 1 ELSE 0 END) AS n_with_comment,
        ROUND(100.0 * SUM(CASE WHEN r.review_comment_message IS NOT NULL AND LENGTH(TRIM(r.review_comment_message)) > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_comment,
        ROUND(AVG(LENGTH(r.review_comment_message)), 1) AS avg_comment_length
    FROM master_orders_sql m
    JOIN order_reviews r ON m.order_id = r.order_id
    WHERE m.is_delivered = 1 AND m.is_late = 0
    GROUP BY review_group
""", conn)
comment_rates


,review_group,n,n_with_comment,pct_with_comment,avg_comment_length
0,bad,8187,6411,78.3,103.3
1,good,73388,25377,34.6,53.8
2,mid,7086,3027,42.7,83.5


**Strong signal:** on-time customers who leave a bad review write a comment **78.3%** of the time (vs. **34.6%** for good reviews), and those comments run almost **2x longer** (103 vs. 54 characters average). This isn't a low-effort star rating — these customers are specifically motivated to explain what went wrong. That's exactly the population worth mining.

In [6]:
bad_comments = pd.read_sql("""
    SELECT r.review_comment_message
    FROM master_orders_sql m
    JOIN order_reviews r ON m.order_id = r.order_id
    WHERE m.is_delivered = 1 AND m.is_late = 0 AND r.review_score <= 2
      AND r.review_comment_message IS NOT NULL AND LENGTH(TRIM(r.review_comment_message)) > 0
""", conn)

good_comments = pd.read_sql("""
    SELECT r.review_comment_message
    FROM master_orders_sql m
    JOIN order_reviews r ON m.order_id = r.order_id
    WHERE m.is_delivered = 1 AND m.is_late = 0 AND r.review_score >= 4
      AND r.review_comment_message IS NOT NULL AND LENGTH(TRIM(r.review_comment_message)) > 0
""", conn)

print(f'Bad-review comments (on-time):  {len(bad_comments):,}')
print(f'Good-review comments (on-time): {len(good_comments):,}')


Bad-review comments (on-time):  6,411
Good-review comments (on-time): 25,377


In [7]:
# Portuguese stopword list -- common function words that would otherwise dominate any word count.
# Includes accented forms (não, até, está, porém, pois) since the tokenizer keeps accents.
PT_STOPWORDS = set("""
de a o que e do da em um para com nao não uma os no se na por mais as dos como mas
foi ao ele das tem seu sua ou ser quando muito ha nos ja esta está eu tambem so
pelo pela ate até isso ela entre era depois sem mesmo aos ter seus quem nas me esse
eles voce você essa num nem suas meu minha numa pelos pelas isto aquele aquela
esses essas dele deles nossa nosso ainda todo toda outro outra onde
sao são foram este estes estas produto pra pro veio recebi comprei apenas
estou pois porem porém quero fiz vou ter meus minhas nao tao tão la lá aqui
""".split())

def top_words(comments_df, col='review_comment_message', top_n=20):
    counter = Counter()
    for text in comments_df[col].dropna():
        words = re.findall(r'[a-zà-ú]{3,}', text.lower())
        words = [w for w in words if w not in PT_STOPWORDS]
        counter.update(words)
    return counter.most_common(top_n)

print('Top words in ON-TIME BAD-review comments:')
for word, count in top_words(bad_comments):
    print(f'  {word:20s} {count}')


Top words in ON-TIME BAD-review comments:
  entregue             770
  chegou               652
  entrega              457
  pedido               444
  compra               424
  dois                 413
  qualidade            403
  produtos             383
  loja                 372
  site                 343
  prazo                311
  diferente            291
  duas                 286
  contato              269
  nota                 266
  agora                263
  pedi                 258
  lannister            257
  troca                257
  errado               252


In [8]:
print('Top words in ON-TIME GOOD-review comments (for contrast):')
for word, count in top_words(good_comments):
    print(f'  {word:20s} {count}')


Top words in ON-TIME GOOD-review comments (for contrast):
  prazo                6877
  antes                5218
  entrega              4466
  chegou               3892
  bom                  3846
  recomendo            3727
  bem                  2620
  qualidade            2082
  entregue             2074
  tudo                 2006
  excelente            1720
  ótimo                1695
  super                1595
  loja                 1393
  gostei               1392
  rápida               1334
  dentro               1138
  compra               1078
  boa                  989
  rápido               901


**Reading this, the contrast is sharper than expected:**

- The **good-review list is dominated by speed and sentiment words**: *antes* (before schedule), *rápida/rápido* (fast), *dentro* (within [the deadline]), plus pure praise — *bom, recomendo, excelente, ótimo, gostei*.
- The **bad-review list has almost none of those** — instead: *dois/duas* (two — counting units, matching Section 2's multi-item finding directly), *diferente* (different — a mismatch), *errado* (wrong), *troca* (exchange/return), *contato* (contact — chasing support), *pedi* (what I ordered/asked for).

That pattern — counting words, "different," "wrong," and "exchange" replacing speed and praise words — lines up precisely with Section 2's quantitative finding: **on-time bad reviews are a product/fulfillment-mismatch problem (wrong item, item count, quality vs. expectation), not a shipping-speed or shipping-cost complaint.** This is a simple word-frequency pass, not a trained topic model — a reasonable first cut, and proper NLP topic modeling on this comment set would be the natural next step to confirm and refine these categories.

**One more thing this scan surfaced, worth flagging on its own:** the token `lannister` (257 occurrences) turned out not to be noise — checking the raw text shows Olist has anonymized real store/seller names by substituting *Game of Thrones* house names (`lannister`, `targaryen`, ...) in the comment text. That's a genuine data-quality quirk: any downstream NLP on these comments should treat these as anonymized entity placeholders rather than real words, or they'll pollute topic models and word counts exactly the way they almost did here.

## Section 4 — A revenue-loss channel the earlier notebooks never covered: orders that never deliver at all

Notebooks 3–4 only ever looked at `is_delivered = 1` orders — late or on-time. But roughly 3% of orders never reach "delivered" status at all: canceled, stuck in processing, marked unavailable. That's not "late" — it's a full loss, and it was invisible in every revenue-at-risk number reported so far.

In [9]:
status_breakdown = pd.read_sql("""
    SELECT order_status, COUNT(*) AS n_orders
    FROM orders
    GROUP BY order_status
    ORDER BY n_orders DESC
""", conn)
status_breakdown


,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [10]:
non_delivered_revenue = pd.read_sql("""
    WITH order_rev AS (
        SELECT order_id, SUM(price) + SUM(freight_value) AS order_revenue
        FROM order_items
        GROUP BY order_id
    )
    SELECT
        o.order_status,
        COUNT(*) AS n_orders,
        ROUND(SUM(COALESCE(r.order_revenue, 0)), 2) AS total_revenue_at_stake,
        ROUND(AVG(r.order_revenue), 2) AS avg_order_revenue
    FROM orders o
    LEFT JOIN order_rev r ON o.order_id = r.order_id
    WHERE o.order_status != 'delivered'
    GROUP BY o.order_status
    ORDER BY total_revenue_at_stake DESC
""", conn)
non_delivered_revenue


,order_status,n_orders,total_revenue_at_stake,avg_order_revenue
0,shipped,1107,177129.34,160.15
1,canceled,625,105885.72,229.69
2,processing,301,69394.11,230.55
3,invoiced,314,68988.75,221.12
4,unavailable,609,2140.49,356.75
5,approved,2,241.08,120.54
6,created,5,0.00,NaN


In [11]:
total_lost = non_delivered_revenue['total_revenue_at_stake'].sum()
total_orders_lost = non_delivered_revenue['n_orders'].sum()
print(f'Total non-delivered orders: {total_orders_lost:,} ({total_orders_lost/99441*100:.2f}% of all orders)')
print(f'Total revenue tied up in never-delivered orders: R$ {total_lost:,.2f}')


Total non-delivered orders: 2,963 (2.98% of all orders)
Total revenue tied up in never-delivered orders: R$ 423,779.49


**This is a genuinely new finding, not a restatement of the late-delivery problem:** ~2,963 orders (about 3% of all orders) never reach the customer at all, representing roughly **R$424,000** in revenue that's either lost outright (canceled, unavailable) or stuck in limbo (processing, invoiced — paid for but never shipped). `shipped` and `canceled` carry the largest dollar exposure. A quick check for concentration by product category (below) shows no single category dominates cancellations — they're roughly proportional to each category's overall order volume, so this looks like a general operational leak rather than a category-specific problem.

In [12]:
cancel_by_category = pd.read_sql("""
    WITH item_cat AS (
        SELECT
            oi.order_id,
            p.product_category_name_english AS category,
            ROW_NUMBER() OVER (PARTITION BY oi.order_id ORDER BY oi.price DESC) AS rn
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
    )
    SELECT ic.category, COUNT(*) AS n_canceled
    FROM orders o
    JOIN item_cat ic ON o.order_id = ic.order_id AND ic.rn = 1
    WHERE o.order_status = 'canceled'
    GROUP BY ic.category
    ORDER BY n_canceled DESC
    LIMIT 8
""", conn)
cancel_by_category


,category,n_canceled
0,sports_leisure,47
1,housewares,37
2,health_beauty,35
3,computers_accessories,35
4,toys,31
5,furniture_decor,24
6,auto,24
7,watches_gifts,20


## Section 5 — A hypothesis tested and ruled out: payment approval delay

Boleto (Brazilian bank-slip payment) is known to take longer to confirm than credit card. Worth testing directly: does that approval delay eat into the delivery window enough to explain part of the lateness or review problem?

In [13]:
approval_by_payment = pd.read_sql("""
    SELECT
        p.payment_type,
        COUNT(DISTINCT o.order_id) AS n_orders,
        ROUND(AVG((julianday(o.order_approved_at) - julianday(o.order_purchase_timestamp)) * 24), 2) AS avg_approval_hours
    FROM orders o
    JOIN order_payments p ON o.order_id = p.order_id
    WHERE o.order_approved_at IS NOT NULL
    GROUP BY p.payment_type
    ORDER BY avg_approval_hours DESC
""", conn)
approval_by_payment


,payment_type,n_orders,avg_approval_hours
0,boleto,19754,33.12
1,debit_card,1528,9.54
2,voucher,3793,8.64
3,credit_card,76449,4.60


In [14]:
approval_effect = pd.read_sql("""
    SELECT
        (julianday(o.order_approved_at) - julianday(o.order_purchase_timestamp)) * 24 AS approval_hours,
        o.delivery_time_days,
        o.is_late
    FROM orders o
    WHERE o.order_approved_at IS NOT NULL AND o.is_delivered = 1
""", conn)

corr_approval_delivery = approval_effect['approval_hours'].corr(approval_effect['delivery_time_days'])
corr_approval_late = approval_effect['approval_hours'].corr(approval_effect['is_late'])

review_by_payment = pd.read_sql("""
    SELECT p.payment_type, ROUND(AVG(m.review_score), 2) AS avg_review_score, COUNT(*) AS n
    FROM master_orders_sql m
    JOIN order_payments p ON m.order_id = p.order_id
    WHERE m.is_delivered = 1 AND m.review_score IS NOT NULL
    GROUP BY p.payment_type
    ORDER BY n DESC
""", conn)

print(f'Correlation, approval_hours vs delivery_time_days: {corr_approval_delivery:.3f}')
print(f'Correlation, approval_hours vs is_late:            {corr_approval_late:.3f}')
print()
print('Average review score by payment type:')
review_by_payment


Correlation, approval_hours vs delivery_time_days: 0.080
Correlation, approval_hours vs is_late:            0.029

Average review score by payment type:


,payment_type,avg_review_score,n
0,credit_card,4.15,74087
1,boleto,4.16,19062
2,voucher,4.12,5452
3,debit_card,4.24,1479


**Verdict: ruled out as a driver of lateness or reviews, but still worth fixing on its own.** Boleto really does take ~7x longer to confirm than credit card (**33.1 hours vs. 4.6 hours**) — that part of the hypothesis is true. But it doesn't propagate into the outcomes this project cares about: correlation with total delivery time is negligible (**0.08**), correlation with being late is essentially zero (**0.02**), and average review score is flat across payment types (**4.12–4.24**, no meaningful spread). 

Reporting this as a **null result** rather than dropping it silently matters: it means Olist's delivery-estimate logic likely already accounts for payment method, so a recommendation to "speed up boleto confirmation to fix lateness" would be based on a plausible-sounding but false premise. It's still a legitimate, separate operational efficiency opportunity — just not a lever for *this* problem.

## Section 6 — Updated findings summary

**What Notebooks 3–4 already established:**
- 8.1% platform-wide late-delivery rate; ~15% of revenue tied to 1–2 star orders
- Specific underperforming sellers, a northeast Brazil regional pattern, and a state × category revenue-at-risk ranking

**What this notebook adds:**
1. **The lateness → reviews link is real but partial** (order-level correlation −0.27 to −0.33) — confirmed with the correct (order-level, not aggregate) test
2. **On-time bad reviews concentrate in higher-value, multi-item orders** (+21% price, +27% items vs. good reviews) — a product/fulfillment-quality issue, not a shipping-cost or shipping-speed one (freight ratio ruled out)
3. **Comment behavior itself is a signal**: on-time unhappy customers write detailed complaints 78% of the time (vs. 35% for happy ones). A word-frequency scan sharpens this further — good reviews skew toward speed/praise words (*antes, rápida, recomendo, excelente*), bad reviews skew toward mismatch language (*dois/duas, diferente, errado, troca*) — confirming a product/fulfillment issue, not a shipping one. The scan also surfaced a genuine data quirk: store names in the comment text are anonymized as *Game of Thrones* house names (e.g. `lannister`), which any follow-up NLP work needs to filter as placeholders rather than real words
4. **A previously invisible revenue-loss channel**: ~3% of all orders (R$424K) never reach "delivered" at all — separate from, and additive to, the late-delivery revenue-at-risk figure
5. **Payment approval delay was tested and ruled out** as a driver of lateness or reviews, despite boleto genuinely taking 7x longer to confirm — an honest null result, not a gap in the analysis

**Combined, more complete revenue-at-risk picture:** ~R$2.3M (low-rated orders) + ~R$1.35M (late orders, overlapping with the above) + ~R$424K (never-delivered orders) — the last of which was entirely outside the scope of the original framing.


In [15]:
conn.close()
print('Investigation complete.')


Investigation complete.
